In [1]:
from pyspark.sql import SparkSession, Row

In [2]:
from pyspark.sql.functions import col, when, count, lit

In [3]:
#import pandas as pd
#pd.set_option('display.max_colwidth', None) # Display full content of cells

In [4]:
spark = SparkSession.builder.appName('pipeline').getOrCreate()

In [5]:
surveySchema = spark.read.option('header', True).csv('survey_results_schema.csv')

In [6]:
# surveySchema.show(n=surveyResultsSchemaDF.count(),truncate=False) ## Action, dont touch

In [7]:
surveyResults = spark.read.option('header', True).csv('survey_results_public.csv')

In [8]:
surveySchema.printSchema()

root
 |-- qid: string (nullable = true)
 |-- qname: string (nullable = true)
 |-- question: string (nullable = true)
 |-- force_resp: string (nullable = true)
 |-- type: string (nullable = true)
 |-- selector: string (nullable = true)



In [9]:
surveyResults.printSchema()

root
 |-- ResponseId: string (nullable = true)
 |-- MainBranch: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Employment: string (nullable = true)
 |-- RemoteWork: string (nullable = true)
 |-- Check: string (nullable = true)
 |-- CodingActivities: string (nullable = true)
 |-- EdLevel: string (nullable = true)
 |-- LearnCode: string (nullable = true)
 |-- LearnCodeOnline: string (nullable = true)
 |-- TechDoc: string (nullable = true)
 |-- YearsCode: string (nullable = true)
 |-- YearsCodePro: string (nullable = true)
 |-- DevType: string (nullable = true)
 |-- OrgSize: string (nullable = true)
 |-- PurchaseInfluence: string (nullable = true)
 |-- BuyNewTool: string (nullable = true)
 |-- BuildvsBuy: string (nullable = true)
 |-- TechEndorse: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Currency: string (nullable = true)
 |-- CompTotal: string (nullable = true)
 |-- LanguageHaveWorkedWith: string (nullable = true)
 |-- LanguageWantToWorkWith:

In [10]:
surveySelected = surveyResults.select('Country', 'Age')
surveyAgeCleaned = surveySelected.withColumn("AgeClean", when(col("Age").isNull(), "Unknown").when(col('Age') == "Prefer not to say", "Unknown").otherwise(col("Age")))

In [11]:
surveyAgeCleaned.show()

+--------------------+------------------+------------------+
|             Country|               Age|          AgeClean|
+--------------------+------------------+------------------+
|United States of ...|Under 18 years old|Under 18 years old|
|United Kingdom of...|   35-44 years old|   35-44 years old|
|United Kingdom of...|   45-54 years old|   45-54 years old|
|              Canada|   18-24 years old|   18-24 years old|
|              Norway|   18-24 years old|   18-24 years old|
|United States of ...|Under 18 years old|Under 18 years old|
|United States of ...|   35-44 years old|   35-44 years old|
|          Uzbekistan|   18-24 years old|   18-24 years old|
|United Kingdom of...|   45-54 years old|   45-54 years old|
|              Serbia|   35-44 years old|   35-44 years old|
|United States of ...|   35-44 years old|   35-44 years old|
|              Poland|   45-54 years old|   45-54 years old|
|United States of ...|   35-44 years old|   35-44 years old|
|         Philippines|  

In [12]:
surveyAgeCleaned.filter(surveyAgeCleaned['AgeClean'] == "Unknown").show()

+--------------------+-----------------+--------+
|             Country|              Age|AgeClean|
+--------------------+-----------------+--------+
|           Australia|Prefer not to say| Unknown|
|              Turkey|Prefer not to say| Unknown|
|United Kingdom of...|Prefer not to say| Unknown|
|             Hungary|Prefer not to say| Unknown|
|            Thailand|Prefer not to say| Unknown|
|United Kingdom of...|Prefer not to say| Unknown|
|              Israel|Prefer not to say| Unknown|
|United States of ...|Prefer not to say| Unknown|
|         Afghanistan|Prefer not to say| Unknown|
|United States of ...|Prefer not to say| Unknown|
|United States of ...|Prefer not to say| Unknown|
|United States of ...|Prefer not to say| Unknown|
|              Poland|Prefer not to say| Unknown|
|United States of ...|Prefer not to say| Unknown|
|United Kingdom of...|Prefer not to say| Unknown|
|United States of ...|Prefer not to say| Unknown|
|         Afghanistan|Prefer not to say| Unknown|


In [13]:
bucketDF = surveyAgeCleaned.groupBy("Country").pivot("Age").agg(count(lit(1)))
bucketDF.printSchema()

root
 |-- Country: string (nullable = true)
 |-- 18-24 years old: long (nullable = true)
 |-- 25-34 years old: long (nullable = true)
 |-- 35-44 years old: long (nullable = true)
 |-- 45-54 years old: long (nullable = true)
 |-- 55-64 years old: long (nullable = true)
 |-- 65 years or older: long (nullable = true)
 |-- Prefer not to say: long (nullable = true)
 |-- Under 18 years old: long (nullable = true)



In [14]:
bucketDF.write.option('header', True).mode("overwrite").csv('ageGroupsOfCountry.csv')

In [15]:
from pyspark.sql.functions import split, explode, trim, lower

In [16]:
langDF = surveyResults.select(col("ResponseID"), col("Country"), col("Age"), col("LanguageHaveWorkedWith"))

In [17]:
langDF

DataFrame[ResponseID: string, Country: string, Age: string, LanguageHaveWorkedWith: string]

In [20]:
langDFCleaned = langDF.withColumn("AgeClean", when(col("Age").isNull(), "Unknown").when(col('Age') == "Prefer not to say", "Unknown").otherwise(col("Age")))

In [21]:
langDFCleaned.show()

+----------+--------------------+------------------+----------------------+------------------+
|ResponseID|             Country|               Age|LanguageHaveWorkedWith|          AgeClean|
+----------+--------------------+------------------+----------------------+------------------+
|         1|United States of ...|Under 18 years old|                    NA|Under 18 years old|
|         2|United Kingdom of...|   35-44 years old|  Bash/Shell (all s...|   35-44 years old|
|         3|United Kingdom of...|   45-54 years old|                    C#|   45-54 years old|
|         4|              Canada|   18-24 years old|  C;C++;HTML/CSS;Ja...|   18-24 years old|
|         5|              Norway|   18-24 years old|  C++;HTML/CSS;Java...|   18-24 years old|
|         6|United States of ...|Under 18 years old|  Bash/Shell (all s...|Under 18 years old|
|         7|United States of ...|   35-44 years old|                     R|   35-44 years old|
|         8|          Uzbekistan|   18-24 years ol

In [22]:
dfCLang = langDFCleaned.withColumn("Language", explode(split(col("LanguageHaveWorkedWith"), ";")))
dfCLang = dfCLang.withColumn("Language", trim(col("Language")))

In [23]:
dfCLang.show()

+----------+--------------------+------------------+----------------------+------------------+--------------------+
|ResponseID|             Country|               Age|LanguageHaveWorkedWith|          AgeClean|            Language|
+----------+--------------------+------------------+----------------------+------------------+--------------------+
|         1|United States of ...|Under 18 years old|                    NA|Under 18 years old|                  NA|
|         2|United Kingdom of...|   35-44 years old|  Bash/Shell (all s...|   35-44 years old|Bash/Shell (all s...|
|         2|United Kingdom of...|   35-44 years old|  Bash/Shell (all s...|   35-44 years old|                  Go|
|         2|United Kingdom of...|   35-44 years old|  Bash/Shell (all s...|   35-44 years old|            HTML/CSS|
|         2|United Kingdom of...|   35-44 years old|  Bash/Shell (all s...|   35-44 years old|                Java|
|         2|United Kingdom of...|   35-44 years old|  Bash/Shell (all s.

In [28]:
ageGroup = dfCLang.select(col("ResponseID"), col("Country"), col("AgeClean"), col("Language"))

In [29]:
ageGroup.show()

+----------+--------------------+------------------+--------------------+
|ResponseID|             Country|          AgeClean|            Language|
+----------+--------------------+------------------+--------------------+
|         1|United States of ...|Under 18 years old|                  NA|
|         2|United Kingdom of...|   35-44 years old|Bash/Shell (all s...|
|         2|United Kingdom of...|   35-44 years old|                  Go|
|         2|United Kingdom of...|   35-44 years old|            HTML/CSS|
|         2|United Kingdom of...|   35-44 years old|                Java|
|         2|United Kingdom of...|   35-44 years old|          JavaScript|
|         2|United Kingdom of...|   35-44 years old|              Python|
|         2|United Kingdom of...|   35-44 years old|          TypeScript|
|         3|United Kingdom of...|   45-54 years old|                  C#|
|         4|              Canada|   18-24 years old|                   C|
|         4|              Canada|   18

In [30]:
ageGroupCOnly = ageGroup.filter(trim(lower(col("Language"))) == "c")

In [31]:
ageGroupCOnly.show()

+----------+--------------------+------------------+--------+
|ResponseID|             Country|          AgeClean|Language|
+----------+--------------------+------------------+--------+
|         4|              Canada|   18-24 years old|       C|
|        12|              Poland|   45-54 years old|       C|
|        15|            Bulgaria|   25-34 years old|       C|
|        40|        Saudi Arabia|Under 18 years old|       C|
|        47|United States of ...|   35-44 years old|       C|
|        53|             Germany|   18-24 years old|       C|
|        56|             Germany|   55-64 years old|       C|
|        59|               India|   18-24 years old|       C|
|        63|             Algeria|   18-24 years old|       C|
|        66|Iran, Islamic Rep...|   18-24 years old|       C|
|        67|         Switzerland|   18-24 years old|       C|
|        73|            Pakistan|   18-24 years old|       C|
|        75|         Switzerland|   55-64 years old|       C|
|       

In [32]:
grouped = ageGroupCOnly.groupBy("Country", "AgeClean").agg(count("*").alias("count"))

In [33]:
grouped.show()

+--------------------+------------------+-----+
|             Country|          AgeClean|count|
+--------------------+------------------+-----+
|              France|   55-64 years old|   26|
|         Netherlands|   45-54 years old|   26|
|             Uruguay|   18-24 years old|    2|
|             Croatia|   25-34 years old|   14|
|             Albania|   35-44 years old|    1|
|             Georgia|   35-44 years old|    3|
|            Slovakia|   45-54 years old|    4|
|United Kingdom of...|   55-64 years old|   49|
|             Denmark|   18-24 years old|   20|
|            Viet Nam|   18-24 years old|   44|
|              Serbia|   25-34 years old|    5|
|             Germany|   25-34 years old|  356|
|United Kingdom of...|   45-54 years old|   74|
|              Canada|   55-64 years old|   25|
|  Dominican Republic|   18-24 years old|    2|
|           Australia|   18-24 years old|   62|
|               Italy|   45-54 years old|   46|
|             Croatia|   18-24 years old

In [34]:
pivoted = grouped.groupBy("Country").pivot("AgeClean").sum("count").fillna(0)

In [35]:
pivoted.show()

+------------------+---------------+---------------+---------------+---------------+---------------+-----------------+------------------+-------+
|           Country|18-24 years old|25-34 years old|35-44 years old|45-54 years old|55-64 years old|65 years or older|Under 18 years old|Unknown|
+------------------+---------------+---------------+---------------+---------------+---------------+-----------------+------------------+-------+
|     Côte d'Ivoire|              0|              2|              0|              0|              0|                0|                 0|      0|
|              Chad|              0|              1|              0|              0|              0|                0|                 0|      0|
|          Paraguay|              2|              1|              0|              0|              0|                0|                 0|      0|
|             Yemen|              1|              1|              0|              0|              0|                0|      

In [36]:
pivoted.write.option("headers", True).csv('CUsers.csv')